# Eureka! + jwst Parallel Pipeline 
### generate precise, absolute flux-calibrated NIRSpec time series spectra
#### Daphne Broski-Laing -- daphne.broski.laing@virginia.edu


---

#### Motivation:
- Eureka! is great at analyzing the relative variability of TSOs. The jwst pipeline is better equipped to straighten the trace (by resampling the spectra) and produce absolute flux-calibrated spectra, but yields less precise time series data (and requires some manual adjustments to resample TSO spectra). 
- The goal of this pipeline is to take advantage of the strengths of both programs. We use the jwst pipeline to calculate a master calibration factor array that applies uniformly to every exposure so that we generate absolute calibrated fluxes while maintaining the integretiy of the variability detections

---

#### Setup:
- Required packages: Eureka! & jwst pipeline
- The Eureka! pipeline uses rateints (although rate files should also work)
- The jwst pipeline requires rate files in order to resample the spectra (it automatically skips trace resampling if we provide rateint files)


<!-- 
Directories (SCREENSHOTS: https://drive.google.com/drive/folders/1O71zYQmYTgxFJ-O6FBgdNfIL0igMiQCb?usp=drive_link):
- I follow the conventions that are provided for the Eureka! tutorial (https://eurekadocs.readthedocs.io/en/latest/index.html)
- in the "Data" directory, I store the raw data files 
    - Eureka! can work with rate or rateints, but **the jwst pipeline requires rate files**
    - The data is sorted by object and instrument, as shown in the screenshots in the google drive folder
- in the "Data Analysis" directory:
    - I have a directory to store the Eureka .ecf files, the Eureka run_eureka.py script, and the Eureka output (sorted in subdirectories based on object and instrument)
    - I have another subdirectory for the jwst output files
    - NOTE: See the code comments and screenshots of the directories to see how my pipeline clears old output into a backup directory (labeled by date and time)
     -->

Directories: 
- I follow the conventions that are provided for the Eureka! tutorial (https://eurekadocs.readthedocs.io/en/latest/index.html)
- in the "Data" directory, I store the raw data files 
    - Eureka! can work with rate or rateints, but **the jwst pipeline requires rate files**
    - The data is sorted by object and instrument, as shown in the screenshots in the google drive folder
- in the "Data Analysis" directory:
    - I have a directory to store the Eureka .ecf files, the Eureka run_eureka.py script, and the Eureka output (sorted in subdirectories based on object and instrument)
    - I have another subdirectory for the jwst output files
    - NOTE: See the code comments and screenshots of the directories to see how my pipeline clears old output into a backup directory (labeled by date and time)


Eureka! pipeline setup:
- This pipeline only uses Eureka! Stages 1, 2 & 3
<!-- - My Eureka template files are in this folder: (https://drive.google.com/drive/folders/1O71zYQmYTgxFJ-O6FBgdNfIL0igMiQCb?usp=drive_link), which will need to be adjusted to your directory paths. The output structure should follow the conventions of the Eureka! tutorial  -->
- Note that an 'event label', which is defined in the run_eureka.py. Your .ecf files need to be named with the following convention 'S{stage number}_{event label}.ecf' -- for example: S2_VHS1256_nrs2.ecf , where the event label is 'VHS1256_nrs2'
- You will need to adjust the .ecf template files with your directory paths, and you may need to adjust the wavelength and/or pixel windows in Stages 2 & 3 (depending on instrument)
- If you want to run more than just stages 1, 2 & 3 of Eureka, then un-comment the appropriate lines in the run_eureka.py file
---

#### Pipeline output
- At the moment, the important output of this pipeline is all saved in the .H5 file **in the Eureka! output directory**.
- I add new data sets in the .H5 file with the calibrated_optspec, the calibrated_opterr (propagated uncertainties for the spectra), as well as the master_calibration_factor_array
---
### <span style="color:#d916a8"><b>NOTE: update CRDS, as explained in Eureka  </b></span>



# USER INPUT
- Choose which instrument to run the pipeline for
- Choose whether or not you need to run the Eureka! and jwst pipelines (change to false if you've already run the pipeline)
- Choose which exposure type you want to change in the rate.fits files. (leave as-is for general use of the pipeline)
- Update the directories to match your directory setup

In [ ]:
""" 
Object Name
"""

obj_name = "ZTFJ0038+2030"
# obj_name = "WD1032"
# obj_name = "WD1202"
# obj_name = "SDSS1411"
# obj_name = "WD0137"
# obj_name = "VHS1256"


"""
Which instrument to use
"""

# instrument = "G395H_nrs1"
# instrument = "G395H_nrs2"
instrument = "PRISM"


"""
Which steps ofthe pipeline to run/skip (set True/False)
"""
run_from_uncal = True
apply_custom_mask_after_S1 = True
run_jwst_S2 = True
run_eureka = True

# run_from_uncal = False
# apply_custom_mask_after_S1 = False
# run_jwst_S2 = False
# run_eureka = False

"""
Other things that you can change but probably won't need to 

# notes for high cadence:
- don't run from uncal, use the S1 files already in the directory
"""

high_cadence = True
if high_cadence == True:
    run_from_uncal = False # override previous selection for high cadence data


diy_rate_files = False # # IGNORE -- (I only created this for troubleshooting a specific issue)



# what rate.fits header EXP_TYPE entry to change 

# DEFAULTS
old_exp_type='NRS_BRIGHTOBJ'
new_exp_type='NRS_FIXEDSLIT'


### Directories

What you need:
- where you want to save the output from the jwst pipeline
- name of environment to run Eureka! on
- directory where all Eureka! pipeline files (.ecf files, python script, etc.) are stored
- name of the python script to run the Eureka! pipeline stage 1 (just converting raw data to rate/rateints)
- name of the python script to run the Eureka! TSO processing (stages 2, 3)

In [ ]:

if instrument == "PRISM":

    # where you want the jwst S2 data to be stored
    jwst_S2_output_dir = f'/Users/rqy2qb/Library/CloudStorage/OneDrive-Personal/Documents/Research/TSO_Analysis/Parallel_pipeline_for_Lael/DataAnalysis/JWST/{obj_name}/{obj_name}_jwst/PRISM'

    # Eureka things:
    eureka_env_name = "Eureka_April" # eureka conda environment
    S1_only_eureka_script_name = "run_eureka_S1.py" 
    eureka_script_name = "run_eureka.py"

    if high_cadence ==False:
        eureka_commands_directory = f"/Users/rqy2qb/Library/CloudStorage/OneDrive-Personal/Documents/Research/TSO_Analysis/Parallel_pipeline_for_Lael/DataAnalysis/JWST/{obj_name}/{obj_name}_Eureka/PRISM" # where you have all your eureka pipeline files (.ecf, the python script, etc.)
    elif high_cadence == True:
        eureka_commands_directory = f"/Users/rqy2qb/Library/CloudStorage/OneDrive-Personal/Documents/Research/TSO_Analysis/Parallel_pipeline_for_Lael/DataAnalysis/JWST/{obj_name}/{obj_name}_Eureka_high_cadence/PRISM" # where you have all your eureka pipeline files (.ecf, the python script, etc.)

  
    # if diy_rate_files == True:
    #     eureka_commands_directory = eureka_commands_directory.replace("/PRISM", "_diy_rate_files/PRISM")
    #     jwst_S2_output_dir = jwst_S2_output_dir.replace("/PRISM", "_diy_rate_files/PRISM")



### IGNORE BELOW (unless you use this pipeline for data with other instruments :) )

# if instrument == "G395H_nrs1":

#     # where you want the jwst S2 data to be stored
#     jwst_S2_output_dir = f'/Users/rqy2qb/DataAnalysis/JWST/{obj_name}/{obj_name}_jwst/G395H_nrs1'
#     # eureka stuff:
#     eureka_env_name = "Eureka_April" # eureka conda environment
#     eureka_commands_directory = f"/Users/rqy2qb/DataAnalysis/JWST/{obj_name}/{obj_name}_Eureka/G395H_nrs1/" # where you have all your eureka pipeline files (.ecf, the python script, etc.)
#     S1_only_eureka_script_name = "run_eureka_S1.py" 
#     eureka_script_name = "run_eureka.py" 



# if instrument == "G395H_nrs2":


#     # where you want the jwst S2 data to be stored
#     jwst_S2_output_dir = f'/Users/rqy2qb/DataAnalysis/JWST/{obj_name}/{obj_name}_jwst/G395H_nrs2'

#     # eureka stuff:
#     eureka_env_name = "Eureka_April" # eureka conda environment
#     eureka_commands_directory = f"/Users/rqy2qb/DataAnalysis/JWST/{obj_name}/{obj_name}_Eureka/G395H_nrs2/" # where you have all your eureka pipeline files (.ecf, the python script, etc.)
#     S1_only_eureka_script_name = "run_eureka_S1.py" 
#     eureka_script_name = "run_eureka.py" 




In [ ]:
# print the instrument and selected directories
print(f"""
Instrument:                 {instrument} 
Running jwst pipeline?      {run_jwst_S2}
Running Eureka pipeline?    {run_eureka}

jwst S2 output directory:   {jwst_S2_output_dir}
Eureka commmands directory: {eureka_commands_directory}
Eureka script name:         {eureka_script_name}
       """)

### Imports

In [ ]:
# imports
import os
import shutil
from os import path, makedirs
import glob
import matplotlib.pyplot as plt
from astropy.io import fits
from jwst.pipeline import Spec2Pipeline       #calwebb_spec2
import subprocess
from datetime import datetime
import h5py
import numpy as np
from scipy.interpolate import interp1d
# import datetime
import matplotlib.cm as cm
import time

## 0) Start from the uncalibrated jwst data

Process the uncal data with Eureka! to generate rate.fits and rateints.fits

In [ ]:
# Define directories
directories_to_backup = ["Stage1"]
backup_directory = os.path.join(eureka_commands_directory, "eureka_data_backup")

# Check if the pipeline should be run
if run_from_uncal == True:
    print('\nRunning Eureka Stage 1')
    # Change to the target directory
    os.chdir(eureka_commands_directory)

    # Create the backup directory if it doesn't exist
    if not os.path.exists(backup_directory):
        os.makedirs(backup_directory)
        print(f"Created backup directory: {backup_directory}")

    # Get the current date and time to create a timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")

    # Move specified directories' contents to the backup directory with a timestamp
    for dir_name in directories_to_backup:
        dir_path = os.path.join(eureka_commands_directory, dir_name)
        if os.path.exists(dir_path):
            # Define the new backup subdirectory name with the timestamp
            dest_path = os.path.join(backup_directory, f"{dir_name}_{timestamp}")
            os.makedirs(dest_path, exist_ok=True)

            # Move each item inside the directory to the timestamped backup subdirectory
            for item in os.listdir(dir_path):
                shutil.move(os.path.join(dir_path, item), dest_path)
            print(f"Moved contents of {dir_name} to {dest_path}")

            # Delete the original directory after moving its contents
            shutil.rmtree(dir_path)
            print(f"Deleted original directory: {dir_path}")
        else:
            print(f"Directory does not exist: {dir_path}")

    # Construct the command
    command = ["conda", "run", "-n", eureka_env_name, "python", S1_only_eureka_script_name]

    # Run the command and capture output
    result = subprocess.run(command, capture_output=True, text=True)

    # Print the output and error (if any)jw
    print("Output:", result.stdout)
    print("Error:", result.stderr)

else:
    print("Assuming eureka S1 has already been run, moving on")

In [ ]:
# the rate and rateints files should be saved in the stage 1 directory
# go into the stage 1 directory, enter the run1 folder (there will only be one folder in Stage1 directory)
# the rate and rateints files should be in there

os.chdir(os.path.join(eureka_commands_directory, "Stage1"))
# get the run1 folder
run1_folders = glob.glob("*run1") # there should only be one folder
run1_dir = run1_folders[0]
run1_path = os.path.join(eureka_commands_directory, "Stage1", run1_dir)
print(f"Run1 directory: {run1_path}")

# # now change the name of the run1 folder to S1_{obj_name}_{instrument}_run1
# new_run1_dir_name = f"S1_{obj_name}_{instrument}_run1"
# os.rename(run1_dir, new_run1_dir_name)
# run1_dir = new_run1_dir_name

# go into the run1 folder
os.chdir(run1_dir)
# get the rate and rateints files
rate_file_paths = glob.glob("*_rate.fits") # need the underscore so it doesn't find the diy-rate.fits files
rateints_file_paths = glob.glob("*rateints.fits")

In [ ]:
# if diy_rate_files == True, then we need to duplicate the rate files but replace the data with the averaged of the associated rateints files
# make a new rate file with the same header as the original rate file, but with the data replaced with the average of the associated rateints files
if diy_rate_files == True:

    # get the rate file names
    rate_file_names = [os.path.basename(rate_file) for rate_file in rate_file_paths]
    # get the rateints file names
    rateints_file_names = [os.path.basename(rateints_file) for rateints_file in rateints_file_paths]

    # Save the new rate files in the same directory as the original rate files
    new_rate_dir = run1_path  # This is where the original rate files are stored

    # loop through the rate files and their associated rateints files
    for i, rate_file in enumerate(rate_file_paths):
        # get the associated rateints file
        rateints_file = rateints_file_paths[i]
        
        # open the original rate file to create a copy of its structure
        with fits.open(rate_file) as hdul_rate:
            # open the associated rateints file
            with fits.open(rateints_file) as hdul_rateints:
                # Create a new HDUList to store all HDUs
                new_hdul = fits.HDUList()
                
                # Copy the primary HDU exactly
                new_hdul.append(fits.PrimaryHDU(header=hdul_rate[0].header, data=hdul_rate[0].data))
                
                # Process each extension
                for j in range(1, len(hdul_rate)):
                    if isinstance(hdul_rate[j], fits.ImageHDU):
                        # For SCI extension, replace data with averaged data from rateints
                        if hdul_rate[j].header.get('EXTNAME') == 'SCI' and j < len(hdul_rateints):
                            # Average the data from rateints along the integration axis
                            avg_data = np.nanmedian(hdul_rateints[j].data, axis=0)
                            
                            # Create a new ImageHDU with the original header but averaged data
                            new_hdu = fits.ImageHDU(data=avg_data, header=hdul_rate[j].header)
                            new_hdul.append(new_hdu)
                        else:
                            # For other extensions, copy as is
                            new_hdul.append(hdul_rate[j].copy())
                    else:
                        # Copy any non-image HDUs exactly (like tables)
                        new_hdul.append(hdul_rate[j].copy())
                
                # Create the new rate file with _diy-rate added to the end
                new_rate_file_name = os.path.join(new_rate_dir, f"{os.path.basename(rate_file).replace('.fits', '_diy-rate.fits')}")
                new_hdul.writeto(new_rate_file_name, overwrite=True)
                print(f"Created new rate file: {new_rate_file_name}")

# Custom Mask
if apply_custom_mask_after_S1 == True, use the *custom_mask.fits file in the eureka folder 

In [ ]:
if apply_custom_mask_after_S1:
    # Get the custom mask file
    custom_mask_files = glob.glob(os.path.join(eureka_commands_directory, "*custom_mask.fits"))
    
    if not custom_mask_files:
        print("No custom mask file found, skipping")
    else:
        custom_mask_file = custom_mask_files[0]
        print(f"Found custom mask: {custom_mask_file}")

        try:
            mask_data = fits.getdata(custom_mask_file)
            print(f"Mask shape: {mask_data.shape}")
            print(f"Mask data type: {mask_data.dtype}")
            print(f"Unique values in mask: {np.unique(mask_data)}")
            print(f"Number of pixels to mask (value=1): {np.sum(mask_data == 1)}")
            print(f"Number of good pixels (value=0): {np.sum(mask_data == 0)}")
            
            # Check if mask is all 1s (which would mask everything)
            if np.all(mask_data == 1):
                print("WARNING: Custom mask would mask ALL pixels! Skipping mask application.")
                print("Please check your custom mask file - it should have 0s for good pixels and 1s for bad pixels.")
            else:
                # Create a boolean mask for pixels to be masked (where mask_data == 1)
                bad_pixel_mask = (mask_data == 1)
                
                # Apply mask to rate files
                for rate_file in rate_file_paths:
                    rate_file_path = os.path.join(run1_path, rate_file)
                    with fits.open(rate_file_path, mode='update') as hdul:
                        sci_data = hdul['SCI'].data
                        if bad_pixel_mask.shape != sci_data.shape:
                            raise ValueError(f"Mask shape {bad_pixel_mask.shape} does not match SCI shape {sci_data.shape} in {rate_file}")
                        sci_data[bad_pixel_mask] = np.nan
                        print(f"Applied custom mask to {rate_file} - masked {np.sum(bad_pixel_mask)} pixels")

                # Apply mask to rateints files
                for rateints_file in rateints_file_paths:
                    rateints_file_path = os.path.join(run1_path, rateints_file)
                    with fits.open(rateints_file_path, mode='update') as hdul:
                        sci_data = hdul['SCI'].data
                        for i in range(sci_data.shape[0]):
                            if bad_pixel_mask.shape != sci_data[i].shape:
                                raise ValueError(f"Mask shape {bad_pixel_mask.shape} does not match SCI integration shape {sci_data[i].shape} in {rateints_file}")
                            sci_data[i][bad_pixel_mask] = np.nan
                        print(f"Applied custom mask to {rateints_file} - masked {np.sum(bad_pixel_mask)} pixels per integration")
                # add in a 30 second pause to make sure the files are done being written before moving on
                os.sync()
                time.sleep(30)

        
                        
        except Exception as e:
            raise RuntimeError(f"Failed to load custom mask: {e}")
        




## 1) Prepare the rate.fits files for jwst pipeline
Replace "NRS_BRIGHTOBJ" in header with "NRS_FIXEDSLIT" to allow for jwst pipeline to resample the spectrum (this will straighten the trace)

In [ ]:
def update_exp_type(directory_path, suffix='rate.fits', updated_dir_name = f"{obj_name}_rate_files_updated_exp_type_to_{new_exp_type}", old_exp_type='NRS_BRIGHTOBJ', new_exp_type='NRS_FIXEDSLIT'):
    
    """
    Check if a {path}_updated-headers subdirectory exists in the rate_files_dir. 
    If the updated-headers subdirectory doesn't exist, create it, 
    duplicate all rate.fits files into it, and update the EXP_TYPE header in the duplicated files if necessary.

    Regardless of whether the header has already been updated, the files are duplicated into an updated-header subdirectory.
    (The jwst pipeline is run for all files found in the updated-header subdirectory)

    """
    # # Get the name of the provided directory and define the target subdirectory path
    # # directory_name = os.path.basename(os.path.normpath(directory_path))
    updated_dir = os.path.join(directory_path, updated_dir_name)
    
    # Check if the target subdirectory exists
    if not os.path.exists(updated_dir):
        os.makedirs(updated_dir)
        print(f"Created directory: {updated_dir}")
        
        # Loop through all files in the specified directory
        for filename in os.listdir(directory_path):
            # Check if the file ends with the specified suffix
            if filename.endswith(suffix):
                file_path = os.path.join(directory_path, filename)
                new_file_path = os.path.join(updated_dir, filename)
                
                # Copy the fils to the updated subdirectory
                shutil.copy2(file_path, new_file_path)
                print(f"Copied {filename} to {new_file_path}")
                
                # Open the copied FITS file
                with fits.open(new_file_path, mode='update') as hdul:
                    # Check if 'EXP_TYPE' is present and has the value 'NRS_BRIGHTOBJ'
                    if hdul[0].header.get('EXP_TYPE') == old_exp_type:
                        # Update the EXP_TYPE in the primary header
                        hdul[0].header['EXP_TYPE'] = new_exp_type
                        print(f"Updated EXP_TYPE in {new_file_path}")
                    else:
                        print(f"No change needed for {filename} (EXP_TYPE not '{old_exp_type}')")

    else:
        print(f"Directory {updated_dir} already exists. No changes made.")
    
    print("Process complete.")

    # return the directory where the header-updated files are stored
    return updated_dir

In [ ]:
print("\nUpdating Headers")
# use update_exp_type to make a new subdirectory with rate.fits files with appropriate headers

if diy_rate_files == False:
    jwst_header_updated_dir = update_exp_type(run1_path, suffix = '_rate.fits', old_exp_type=old_exp_type, new_exp_type=new_exp_type, updated_dir_name = f"{obj_name}_rate_files_updated_exp_type_to_{new_exp_type}" )
elif diy_rate_files == True:
    jwst_header_updated_dir = update_exp_type(run1_path, suffix = 'diy-rate.fits', old_exp_type=old_exp_type, new_exp_type=new_exp_type, updated_dir_name = f"{obj_name}_diy-rate_files_updated_exp_type_to_{new_exp_type}" )# the rate files are saved in the run1_dir

## 2) Run jwst S2 Pipeline
if instructed in USER INPUT window, run the pipeline 
(the spectra is resampled, which will mitigate our issues with a curved trace)

Note: if there is old data in the jwst pipeline output directory, this data is cleaned out and moved to a backup directory ('/jwst_pipeline_data-backup/')

In [ ]:
def run_S2(data_filename, output_directory):
    """ 
    After adjusting the data file header (to enable the pipeline to resample the spectra), 
    then, run stage 2 of the jwst pipeline

    the desired output: x1d file (but we'll save other files for troubleshooting and future use)
    """

    print(f'output dir: {output_directory}')
    if not path.exists(output_directory):
        makedirs(output_directory)


    # initialize the pipeline instance 
    spec2 = Spec2Pipeline()

    
    spec2.output_dir = output_directory

    #apply coordinate system
    spec2.assign_wcs.skip = False
    spec2.assign_wcs.save_results = True

    #MSA STEP
    spec2.msa_flagging.skip = True
    spec2.msa_flagging.save_results = False

    #1/f noise removal pipeline 1.13.
    #I cant see these parameters at all maybe they are newer
    spec2.nsclean.skip = False
    spec2.nsclean.save_results = True

    #removes the imprint from the MSA structure on the detector
    spec2.imprint_subtract.skip = True
    spec2.imprint_subtract.save_results = False

    #background subtraction step
    spec2.bkg_subtract.skip = False
    spec2.bkg_subtract.save_results = True

    #extract 2d spectra
    spec2.extract_2d.skip = False
    spec2.extract_2d.save_results = True

    #source identification step
    spec2.srctype.skip = False
    spec2.srctype.source_type = 'POINT'  #TSO Observations default to a point source based on several tage
    #spec2.srctype.save_results = True

    #there is another background step but this is for MSA
    spec2.master_background_mos.skip = True
    spec2.master_background_mos.save_results = False

    #wavelength correction for targets that may be offcenter
    #usually for MOS and FS modes
    spec2.wavecorr.skip = True
    spec2.wavecorr.save_results = False

    #straylight correction usally only for MIRI MRS
    spec2.straylight.skip = True
    spec2.straylight.save_results = False

    #for TSO observations the flat field step is run after the extract 2d step
    spec2.flat_field.skip = False
    spec2.flat_field.save_interpolated_flat = True
    spec2.flat_field.save_results = True

    #removing fringes from optics, usually MIRI step
    spec2.fringe.skip = True
    spec2.fringe.save_results = False

    #corrections needed to account for signal loss. Usually not used in TSO data
    spec2.pathloss.skip = True
    spec2.pathloss.save_results = False

    #barshadow step for NIRSpec MSA 
    spec2.barshadow.skip = True
    spec2.barshadow.save_results = False

    #photometric calibration step
    spec2.photom.skip = False
    spec2.photom.save_results = True

    #removing residualfringes from the MIRI data, skipped for TSO data
    spec2.residual_fringe.skip = True
    spec2.residual_fringe.save_results = False

    #replace bad and outlier pixels (described as cosmetic in the docs), not default in TSO observations
    #spec2.pixel_replace.
    spec2.pixel_replace.skip = False
    spec2.pixel_replace.save_results = True

    #resampling spectra to remove distortions typically skipped for TSO observations
    # we have to change the EXP_TYPE in the header if we don't want the pipeline to skip this step for TSO
    spec2.resample_spec.skip = False
    spec2.resample_spec.save_results = True

    #cube building usually skipped in non-IFU modes
    spec2.cube_build.skip = True
    spec2.cube_build.save_results = False

    #extracts from 2d spectra file
    spec2.extract_1d.skip = False
    spec2.extract_1d.save_results = True

    spec2.save_results = True

    spec2.run(data_filename)

In [ ]:
# if the USER INPUT indicates that we should run the S2 pipeline
if run_jwst_S2:
    print('\nRunning jwst pipeline')

    # Step 1: Backup existing files in jwst_S2_output_dir
    if os.path.exists(jwst_S2_output_dir):
        # Create a timestamped folder in jwst_pipeline_data_backup
        timestamp = datetime.now().strftime('%Y-%m-%d_%H%M')
        backup_dir = os.path.join(jwst_S2_output_dir, 'jwst_pipeline_data_backup', f'jwst_S2_output_{timestamp}')
        os.makedirs(backup_dir, exist_ok=True)

        # Move all files in jwst_S2_output_dir to the backup folder, except the backup directory itself
        for item in os.listdir(jwst_S2_output_dir):
            source = os.path.join(jwst_S2_output_dir, item)
            
            # Skip if it's the backup directory
            if os.path.basename(source) == 'jwst_pipeline_data_backup':
                continue

            destination = os.path.join(backup_dir, item)
            shutil.move(source, destination)

        print(f'Contents of {jwst_S2_output_dir} backed up to {backup_dir}')

    # Step 2: Use glob to find all .fits files in the directory with updated headers
    fits_files = glob.glob(os.path.join(jwst_header_updated_dir, '*.fits'))

    # Print the file names to confirm input
    print('INPUT FILES:')
    print("\n".join(os.path.basename(f) for f in fits_files))

    # Step 3: Run Stage 2 of the JWST pipeline for each .fits file
    for file in fits_files:
        run_S2(data_filename=file, output_directory=jwst_S2_output_dir)

else:
    print("Assuming jwst S2 has already been run (as indicated by the USER INPUT window)")
    pass

## 3) Run Eureka! Pipeline (S2 and S3)
Hop into the terminal and run eureka script just like you would in the command prompt 

*The only difference between this code and the default procedure with Eureka! is that I have the pipeline automatically clean out the Eureka! output directory, and store the old data in a backup subdirectory ('/eureka_data_backup/')*

In [ ]:
# Define directories
directories_to_backup = ["Stage2", "Stage3"]
backup_directory = os.path.join(eureka_commands_directory, "eureka_data_backup")

# Check if the pipeline should be run
if run_eureka:
    print('\nRunning Eureka pipeline')
    # Change to the target directory
    os.chdir(eureka_commands_directory)

    # Create the backup directory if it doesn't exist
    if not os.path.exists(backup_directory):
        os.makedirs(backup_directory)
        print(f"Created backup directory: {backup_directory}")

    # Get the current date and time to create a timestamp
    timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")

    # Move specified directories' contents to the backup directory with a timestamp
    for dir_name in directories_to_backup:
        dir_path = os.path.join(eureka_commands_directory, dir_name)
        if os.path.exists(dir_path):
            # Define the new backup subdirectory name with the timestamp
            dest_path = os.path.join(backup_directory, f"{dir_name}_{timestamp}")
            os.makedirs(dest_path, exist_ok=True)

            # Move each item inside the directory to the timestamped backup subdirectory
            for item in os.listdir(dir_path):
                shutil.move(os.path.join(dir_path, item), dest_path)
            print(f"Moved contents of {dir_name} to {dest_path}")

            # Delete the original directory after moving its contents
            shutil.rmtree(dir_path)
            print(f"Deleted original directory: {dir_path}")
        else:
            print(f"Directory does not exist: {dir_path}")

    # Construct the command
    command = ["conda", "run", "-n", eureka_env_name, "python", eureka_script_name]

    # Run the command and capture output
    result = subprocess.run(command, capture_output=True, text=True)

    # Print the output and error (if any)jw
    print("Output:", result.stdout)
    print("Error:", result.stderr)

else:
    print("Assuming eureka S2 and S3 have already been run (as indicated by the USER INPUT window)")

## 4) Now, let's read in the data from both pipelines
### i) How to read in the abs. flux calibrated spectra from jwst (x1d.fits files)
I do this by creating a dictionary called x1ds to store the jwst calibrated spectra

In [ ]:
# Dictionary to store data for each file with an index
x1ds = {}
index_counter = 0  # Initialize an index counter

# Iterate over all files in the directory
for filename in os.listdir(jwst_S2_output_dir):
    # Check if the file ends with "x1d.fits"
    if filename.endswith("x1d.fits"):
        file_path = os.path.join(jwst_S2_output_dir, filename)
        
        # Open the FITS file and keep it open
        hdul = fits.open(file_path)
        
        # Check if the file contains at least one HDU and data table
        if len(hdul) > 1 and isinstance(hdul[1].data, fits.FITS_rec):
            # Extract the observation date and time
            date_obs_str = hdul[0].header.get('DATE-OBS', None)
            time_obs_str = hdul[0].header.get('TIME-OBS', '00:00:00.000')
            
            # Extract start and end times (BSTRTIME and BENDTIME)
            start_time = hdul[0].header.get('BSTRTIME', None)
            end_time = hdul[0].header.get('BENDTIME', None)

            if date_obs_str:
                try:
                    obs_time = datetime.strptime(f"{date_obs_str} {time_obs_str}", '%Y-%m-%d %H:%M:%S.%f')
                except ValueError:
                    obs_time = None  # Handle malformed date strings

                if obs_time:
                    # Extract relevant columns from the table in HDU[1]
                    data_table = hdul[1].data
                    flux = data_table['FLUX'] if 'FLUX' in data_table.columns.names else None
                    wavelength = data_table['WAVELENGTH'] if 'WAVELENGTH' in data_table.columns.names else None
                    flux_error = data_table['FLUX_ERROR'] if 'FLUX_ERROR' in data_table.columns.names else None

                    # Store the data in the dictionary with an index
                    x1ds[filename] = {
                        "index": index_counter,
                        "hdul": hdul,
                        "observation_time": obs_time,
                        "start_time": start_time,
                        "end_time": end_time,
                        "flux": flux,
                        "wavelength": wavelength,
                        "flux_error": flux_error
                    }
                    # index_counter += 1  # Increment the index for the next entry

# Sorting the dictionary by observation time
# Sort the dictionary by observation time
# Sort the dictionary by observation time
x1ds = dict(sorted(x1ds.items(), key=lambda item: item[1]['observation_time']))

# Re-assign index values based on the sorted order
for idx, (fname, data) in enumerate(x1ds.items()):
    data['index'] = idx

# Print the sorted observation times
print("Sorted observation times:")
for fname, data in x1ds.items():
    print(f"{fname}: {data['observation_time']}")

print("Data dictionary loaded and sorted successfully!")
print(f"Total files processed: {len(x1ds)}")


# Function to access data using the index
def get_data_by_index(idx):
    """
    Access data in the dictionary using the assigned index.
    """
    for filename, data in x1ds.items():
        if data['index'] == idx:
            return {
                "filename": filename,
                "observation_time": data['observation_time'],
                "start_time": data['start_time'],
                "end_time": data['end_time'],
                "flux": data['flux'],
                "wavelength": data['wavelength'],
                "flux_error": data['flux_error']
            }
        # else:
        #     print(f"No entry found for index {idx}")
        #     return None

# Example usage: Accessing data for the earliest entry (index 0)
entry = get_data_by_index(1)
if entry:
    print(f"\nAccessing data for index")
    print(f"Filename: {entry['filename']}")
    print(f"Observation Time: {entry['observation_time']}")
    print(f"Start Time: {entry['start_time']}")
    print(f"End Time: {entry['end_time']}")
    print(f"Sample Flux: {entry['flux'][:5] if entry['flux'] is not None else 'No flux data'}")
    print(f"len(Flux): {len(entry['flux'])}")
    print(f"Sample Wavelength: {entry['wavelength'][:5] if entry['wavelength'] is not None else 'No wavelength data'}")
    print(f"Sample Flux Error: {entry['flux_error'][:5] if entry['flux_error'] is not None else 'No flux error data'}")

In [ ]:
get_data_by_index(1)  # Example to access the first entry]

### ii) How to read in the Eureka time series spectra (in an HDF5 file)
This code shows how to access the .H5 file, which stores all of the relevant output data from Eureka!

In [ ]:
# Open the HDF5 file

"""
Explanation of the directory layout:
Each eureka run populates a new folder in each stage folder ("Stage2", "Stage3"). So if you've run S2 twice, the 'Stage2/' directory will have two subdirectories: 'run_1' and 'run_2'

(Note: earlier in the pipeline, I move all old data to a backup folder, so "Stage2" and "Stage3" only contain output from the most recent run.)

Then, in order to get to the Stage3 output data, we have to go through one more intermediate directory (that indicates something about the aperture & background subtraction I think).
So, In order to get to the H5 file, you need to enter the single (unknown) name of the subdirectory in Stage3. 
Then, enter the single (unknown) name of the subdirectory in that run directory to get to the desired data

(long story short: Eureka! Stage3 has two intermediate directories that we have to 'pass through' in order to reach the directory where the actual S3 output is stored)
"""

# Use glob to find the H5 
h5_file_pattern = os.path.join(eureka_commands_directory, 'Stage3', '*', '*', '*SpecData.h5')
h5_file_paths = glob.glob(h5_file_pattern)
# Print the found H5 file paths
print(f"\nFound {len(h5_file_paths)} H5 files matching the pattern '{h5_file_pattern}':")
for h5_file in h5_file_paths:
    print(h5_file)

# print information about the h5 file data structure
if len(h5_file_paths) == 1: # I expect to only find one h5 file in the Eukreka! output
    eureka_output_H5_filename = h5_file_paths[0]
    with h5py.File(eureka_output_H5_filename, 'r') as h5_file:

        def print_structure(name, obj):
            print(f"{name}: {type(obj)}")
        
        h5_file.visititems(print_structure)

else:
    print(f"Error: Expected exactly one h5 file, but found {len(h5_file_paths)}")

## 5) Generate flux calibration factor array and apply to the uncalibrated eureka data

#### STEPS:
i) Generate a list of start and end times for each exposure, and read in the necessary data from the x1d and h5 files

ii) Average those spectra into a single spectrum for each exposure

iii) Generate a calibration factor array from each exposure (uncalibrated/calibrated). Then, interpolate the calibrated spectra (to map to the eureka spectra), average the calibration factors from each exposure to a master calibration factor array
- For each exposure, divide the uncalibrated, averaged spectrum from Eureka H5 file, and divide it by the interpolated+calibrated spectrum to get a calibration factor array

iv) Apply this average calibration factor array to every single integration in the H5 file

NOTE: also - propagate uncertainties through this process

### i) Read in the H5 and x1d files, and make a list of the start and end times for each exposure (which contains multiple integrations)

#### a. Compile a list of the start and end times of the exposures by getting the start & end dates from each exposure (each x1d spectrum output from jwst pipeline)
Note, we don't need to do anything to read in the x1d data because it is already accessible through the x1ds dictionary

In [ ]:
# make lists of start and end times
exposure_start_times = [data['start_time'] for data in x1ds.values() if data['start_time'] is not None]
exposure_end_times = [data['end_time'] for data in x1ds.values() if data['end_time'] is not None]


#### b. Read in the relevant information from the H5 files

In [ ]:
with h5py.File(eureka_output_H5_filename, 'r') as h5_file:
    # Access optspec, time, and wave_1d
    optspec = h5_file['optspec'][:]
    opterr = h5_file['opterr'][:]
    time = h5_file['time'][:]
    eureka_wave_1d = h5_file['wave_1d'][:]



### ii) Sort through the time stamp for each integration, sort the optspec entries based on which integration they're a part of, and generate a single averaged spectrum for each exposure

In [ ]:
# Initialize a list that will store a single, averaged spectrum of each exposure (averaging over integrations)
averaged_exposure_spectra = []
averaged_exposure_spectra_err = []

# Loop through each exposure (one exposure contains all integrations between exposure start and end time)
for exposure_num in range(len(exposure_start_times)):
    start_time = exposure_start_times[exposure_num]
    end_time = exposure_end_times[exposure_num]

    # Initialize an array to hold all of the optspec data for the current exposure
    exposure_integrations = []
    exposure_integrations_err = []

    # Loop through the x1d files and collect optspec data within the specified time range for the current exposure
    for integration_num in range(len(time)):
        if start_time <= time[integration_num] <= end_time:
            # if the integration falls withing the current exposure, add it to the list of exposure_integrations
            exposure_integrations.append(optspec[integration_num])
            exposure_integrations_err.append(opterr[integration_num])
    
    # Now, after we've added all integrations for the current exposure to the exposure_integrations list,
    # get an averaged exposure spectrum and add it to the averaged exposure spectra list.
    averaged_exposure_spectrum = np.nanmean(exposure_integrations, axis=0) # dbl come back
    averaged_exposure_spectra.append(averaged_exposure_spectrum)

   
    # Error propagation:
    exposure_integrations_err = np.array(exposure_integrations_err)
    # divide by N not sqrt(N)
    averaged_exposure_spectrum_err = np.sqrt(np.sum(exposure_integrations_err**2, axis=0)) / len(exposure_integrations)
    averaged_exposure_spectra_err.append(averaged_exposure_spectrum_err)



    # averaged_exposure_spectrum_err = np.sqrt(np.sum(exposure_integrations_err**2, axis=0) / len(exposure_integrations))
    # averaged_exposure_spectra_err.append(averaged_exposure_spectrum_err)



### iii) Generate a calibration factor array from each exposure (uncalibrated/calibrated). Then, interpolate the calibrated spectra (to map to the eureka spectra), average the calibration factors from each exposure to a master calibration factor

In [ ]:
# sanity check:
# ensure that there are the same number of x1d files as averaged exposures from the H5 
if len(x1ds) == len(averaged_exposure_spectra):
    print(f"Number of Exposures: {len(averaged_exposure_spectra)}")
else:
    print("Unexpected Error: the number of uncalibrated spectra (averaged from eureka) is not the same as the number of exposures generated in the jwst pipeline")

In [ ]:
# initialize a list of calibration factors
cal_factors_list = []
cal_factors_err_list = []

# Now, go through the exposures, interpolate the uncalibrated jwst data, and generate calibration factors for each exposure
for exposure_num in range(len(averaged_exposure_spectra)):

    uncalibrated_spectrum = averaged_exposure_spectra[exposure_num] 
    uncalibrated_spectrum_err = averaged_exposure_spectra_err[exposure_num] 
    eureka_wavelengths  = eureka_wave_1d

    calibrated_dict_entry = get_data_by_index(exposure_num)
    calibrated_spectrum = calibrated_dict_entry['flux']
    calibrated_spectrum_err = calibrated_dict_entry['flux_error']
    calibrated_wavelengths = calibrated_dict_entry['wavelength']


    # Interpolate flux and uncertainty

    # Create the interpolation function using the calibrated spectrum
    interp_func_flux = interp1d(calibrated_wavelengths, calibrated_spectrum, kind='linear', fill_value="extrapolate")
    interp_func_err = interp1d(calibrated_wavelengths, calibrated_spectrum_err, kind='linear', fill_value="extrapolate")

    # Interpolate the calibrated spectrum to align with the uncalibrated wavelengths
    interpolated_calibrated_spectrum = interp_func_flux(eureka_wavelengths)
    interpolated_calibrated_spectrum_err = interp_func_err(eureka_wavelengths)


    # Calculate the conversion factor and propagate uncertainty
    cal_factors = uncalibrated_spectrum / interpolated_calibrated_spectrum
    cal_factors_list.append(cal_factors)

    # # DBL UPDATE JUNE 16 - IGNORE CAL FACTORS ERR 
    # # Error propagation
    # # Propagate uncertainties using Bevington's equation
    # cal_factors_err = cal_factors * np.sqrt(
    #     (uncalibrated_spectrum_err / uncalibrated_spectrum) ** 2 +
    #     (interpolated_calibrated_spectrum_err / interpolated_calibrated_spectrum) ** 2
    # )
    # cal_factors_err_list.append(cal_factors_err)


In [ ]:
# plot the jwst output spectra and the interpolated spectra in the same plot

# Set up the figure and axis
fig, ax = plt.subplots(figsize=(10, 5))
# Set the color map
cmap = cm.get_cmap('viridis', len(averaged_exposure_spectra))
# Loop through the exposures and plot each one
for i, exposure_num in enumerate(range(len(averaged_exposure_spectra))):

    
    calibrated_dict_entry = get_data_by_index(exposure_num)
    calibrated_spectrum = calibrated_dict_entry['flux']
    calibrated_spectrum_err = calibrated_dict_entry['flux_error']
    calibrated_wavelengths = calibrated_dict_entry['wavelength']


    # Interpolate flux and uncertainty

    # Create the interpolation function using the calibrated spectrum
    interp_func_flux = interp1d(calibrated_wavelengths, calibrated_spectrum, kind='linear', fill_value="extrapolate")
    interp_func_err = interp1d(calibrated_wavelengths, calibrated_spectrum_err, kind='linear', fill_value="extrapolate")

    # Interpolate the calibrated spectrum to align with the uncalibrated wavelengths
    interpolated_calibrated_spectrum = interp_func_flux(eureka_wavelengths)
    interpolated_calibrated_spectrum_err = interp_func_err(eureka_wavelengths)

    # Plot the uncalibrated spectrum
    ax.plot(eureka_wave_1d, interpolated_calibrated_spectrum, label=f'interpolated calibrated Exposure {exposure_num}', color=cmap(i), alpha=0.5)

    # Plot the interpolated calibrated spectrum
    ax.plot(calibrated_wavelengths, calibrated_spectrum, label=f'Calibrated Exposure {exposure_num}', linestyle='--', color=cmap(i))

# Set labels and title
ax.set_xlabel('Wavelength (microns)')
ax.set_ylabel('Flux (e-/s)')
ax.set_title('JWST Output Spectra vs. Interpolated Calibrated Spectra')
ax.legend()
plt.grid()
plt.show()

In [ ]:
# now, calculate the average of all the cal factors to get a master cal_factors array

# Convert the list to a 2D NumPy array
cal_factors_array = np.array(cal_factors_list)

# Calculate the average calibration factors across all exposures
master_cal_factors = np.mean(cal_factors_array, axis=0) 

### iv) Now, apply the master calibration factor to every integration in the h5 file, and resave the h5 file with a new dataset, which is calibrated_optspec

In [ ]:
with h5py.File(eureka_output_H5_filename, 'r') as h5_file:
    # Access optspec, time, and wave_1d
    optspec = h5_file['optspec'][:]
    opterr = h5_file['opterr'][:]
    time = h5_file['time'][:]
    eureka_wave_1d = h5_file['wave_1d'][:]

    # Ensure that the master calibration factors match the shape of optspec
    if master_cal_factors.shape[0] != optspec.shape[-1]:
        raise ValueError("Shape mismatch: Master calibration factors do not match optspec wavelength dimension.")
    
    
    # Apply the calibration by dividing the optspec by the calibration factors
    calibrated_optspec = optspec / master_cal_factors

    # Error propagation
    # # Step 3: Propagate the error for the calibrated optspec
    # calibrated_opterr = calibrated_optspec * np.sqrt(
    #     (opterr / optspec)**2 + (master_cal_factors_err / master_cal_factors)**2
    # )
    # DBL UPDATE JUNE 16 - IGNORE CAL FACTORS ERR AND JUST TURN OPTERR TO JY
    calibrated_opterr = opterr / master_cal_factors


# Open the original HDF5 file in append mode to add the calibrated dataset
# DZ ADD: propagated errors, calibration factors
with h5py.File(eureka_output_H5_filename, 'a') as h5_file:

    # Check if my parallel pipeline output datasets have already (previously) been saved to this h5 file
    # if so, delete them and replace them with the most current version 
    if 'calibrated_optspec' in h5_file:
        print("Warning: 'calibrated_optspec' already exists. Overwriting it.")
        del h5_file['calibrated_optspec']
    h5_file.create_dataset('calibrated_optspec', data=calibrated_optspec)

    if 'calibrated_opterr' in h5_file:
        print("Warning: 'calibrated_opterr' already exists. Overwriting it.")
        del h5_file['calibrated_opterr']
    h5_file.create_dataset('calibrated_opterr', data=calibrated_opterr)

    if 'master_calibration_factors' in h5_file:
        print("Warning: 'master_calibration_factors' already exists. Overwriting it.")
        del h5_file['master_calibration_factors']
    h5_file.create_dataset('master_calibration_factors', data=master_cal_factors)

    # if 'master_calibration_factors_err' in h5_file:
    #     print("Warning: 'master_calibration_factors_err' already exists. Overwriting it.")
    #     del h5_file['master_calibration_factors_err']
    # h5_file.create_dataset('master_calibration_factors_err', data=master_cal_factors_err)

print(f"Calibrated data added to {eureka_output_H5_filename}")


## 6) Plot the calibrated spectra from the h5 file

In [ ]:
with h5py.File(eureka_output_H5_filename, 'r') as h5_file:
    # Access optspec, time, and wave_1d
    optspec = h5_file['optspec'][:]
    calibrated_optspec = h5_file['calibrated_optspec'][:]
    time = h5_file['time'][:]
    eureka_wave_1d = h5_file['wave_1d'][:]


# Number of integrations
num_integrations = len(calibrated_optspec)

# Set up the colormap (using viridis)
cmap = cm.get_cmap('viridis_r', num_integrations)
colors = cmap(np.linspace(0, 1, num_integrations))

plt.figure(figsize=(10, 6))

# Plot each integration on the same figure with different colors
for integration in range(num_integrations):
    plt.plot(
        eureka_wave_1d,
        calibrated_optspec[integration],
        label=f"Integration {integration + 1}",
        color=colors[integration],
        linewidth = 0.6,
    )

# Uncomment if you want to also plot the interpolated X1D spectrum
# plt.plot(eureka_wave_1d, interpolated_calibrated_spectrum, label="X1D Spectrum (Calibrated, Interpolated)", color="black", linewidth = 0.5)

# Adding labels, legend, and grid
plt.xlabel("Wavelength (microns)")
plt.ylabel("Flux")
plt.title("OptSpec for All Integrations")
# plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1))
plt.grid()
plt.show()

In [ ]:
with h5py.File(eureka_output_H5_filename, 'r') as h5_file:
    # Access optspec, time, and wave_1d
    optspec = h5_file['optspec'][:]
    calibrated_optspec = h5_file['calibrated_optspec'][:]
    time = h5_file['time'][:]
    eureka_wave_1d = h5_file['wave_1d'][:]


# Number of integrations
num_integrations = len(calibrated_optspec)

# Set up the colormap (using viridis)
cmap = cm.get_cmap('viridis_r', num_integrations)
colors = cmap(np.linspace(0, 1, num_integrations))

plt.figure(figsize=(10, 6))

# Plot each integration on the same figure with different colors
for integration in range(num_integrations):
    plt.plot(
        eureka_wave_1d,
        calibrated_optspec[integration],
        label=f"Integration {integration + 1}",
        color=colors[integration],
        linewidth = 0.6
    )

# plt.xlim(4.5,5.5)
# Uncomment if you want to also plot the interpolated X1D spectrum
# plt.plot(eureka_wave_1d, interpolated_calibrated_spectrum, label="X1D Spectrum (Calibrated, Interpolated)", color="black", linewidth = 0.5)
# plt.ylim(0,0.0003)
# Adding labels, legend, and grid
plt.xlabel("Wavelength (microns)")
plt.ylabel("Flux")
plt.title("OptSpec for All Integrations")
# plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1))
plt.grid()
plt.show()

In [ ]:

# Define the number of bins
num_bins = 1 # Adjust as needed
bin_edges = np.linspace(eureka_wave_1d.min(), eureka_wave_1d.max(), num_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Initialize figure
plt.figure(figsize=(10, 6))
vert_shift_interval = 0.2  # Interval for vertical shift between curves

# Define colormap
cmap = plt.get_cmap('turbo')  
colors = [cmap(i / num_bins) for i in range(num_bins)]

# Loop over wavelength bins
for i in range(num_bins):
    # Mask for the current wavelength bin
    bin_mask = (eureka_wave_1d >= bin_edges[i]) & (eureka_wave_1d < bin_edges[i + 1])

    # Compute average flux over the bin for each time integration, ignoring NaNs
    binned_flux = np.nanmean(calibrated_optspec[:, bin_mask], axis=1)
    
    # # Normalize flux (e.g., by mean), handling NaNs
    # normalized_flux = binned_flux / np.nanmean(binned_flux)
    
    # Normalize flux (e.g., by mean)
    normalized_flux = binned_flux / np.mean(binned_flux)
    
    # Apply vertical shift
    vertical_shift = i * vert_shift_interval
    shifted_flux = normalized_flux + vertical_shift
    
    # # Plot phase curve for the bin
    # plt.plot(
    #     time, 
    #     shifted_flux, 
    #     color=colors[i], 
    #     label=f'Bin {i + 1}: {bin_centers[i]:.2f} μm', 
    #     linewidth=0.8,
    # )

    plt.plot(
        time, 
        shifted_flux, 
        color=colors[i], 
        label=f'{bin_centers[i]:.2f} μm', 
        linewidth=0,
        marker = '.', markersize = 2, linestyle='-'
    )

# plt.xlim(time.min(), time.min()+130)  # Set x-axis limits to the full time range

# Customize the plot
plt.xlabel('Time')
plt.ylabel('Fluxes (Normalized & Shifted)')
# plt.ylim(0.9, 2)
plt.title(f'{obj_name} Phase Curve: {instrument}')
plt.ylim(0.75, 1.3)

plt.grid()
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8, title="Wavelength Bin \nCenters")
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:

# Define the number of bins
num_bins = 10 # Adjust as needed
bin_edges = np.linspace(eureka_wave_1d.min(), eureka_wave_1d.max(), num_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Initialize figure
plt.figure(figsize=(20, 8))
vert_shift_interval = 0.25  # Interval for vertical shift between curves

# Define colormap
cmap = plt.get_cmap('turbo')  
colors = [cmap(i / num_bins) for i in range(num_bins)]

# Loop over wavelength bins
for i in range(num_bins):
    # Mask for the current wavelength bin
    bin_mask = (eureka_wave_1d >= bin_edges[i]) & (eureka_wave_1d < bin_edges[i + 1])

    # Compute average flux over the bin for each time integration, ignoring NaNs
    binned_flux = np.nanmean(calibrated_optspec[:, bin_mask], axis=1)
    
    # # Normalize flux (e.g., by mean), handling NaNs
    # normalized_flux = binned_flux / np.nanmean(binned_flux)
    
    # Normalize flux (e.g., by mean)
    normalized_flux = binned_flux / np.mean(binned_flux)
    
    # Apply vertical shift
    vertical_shift = i * vert_shift_interval
    shifted_flux = normalized_flux + vertical_shift
    
    # # Plot phase curve for the bin
    # plt.plot(
    #     time, 
    #     shifted_flux, 
    #     color=colors[i], 
    #     label=f'Bin {i + 1}: {bin_centers[i]:.2f} μm', 
    #     linewidth=0.8,
    # )

    plt.plot(
        time, 
        shifted_flux, 
        color=colors[i], 
        label=f'{bin_centers[i]:.2f} μm', 
        linewidth=0,
        marker = '.', markersize = 7, linestyle='-'
    )

# plt.xlim(time.min(), time.min()+130)  # Set x-axis limits to the full time range

# Customize the plot
plt.xlabel('Time')
plt.ylabel('Fluxes (Normalized & Shifted)')
# plt.ylim(0.9,2)
plt.title(f'{obj_name} Phase Curve: {instrument}')

plt.grid()
plt.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8, title="Wavelength Bin \nCenters")
plt.tight_layout()

# Show the plot
plt.show()

In [ ]:
# Ensure required imports
import matplotlib.pyplot as plt
import numpy as np

# Define the number of bins if not already defined
# num_bins = 8  # Adjust as needed

# Define bin edges and centers if not already defined
bin_edges = np.linspace(eureka_wave_1d.min(), eureka_wave_1d.max(), num_bins + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# Define colors for bins
cmap_bins = plt.get_cmap('turbo')  
colors_bins = [cmap_bins(i / num_bins) for i in range(num_bins)]

# Optional: Create an additional plot showing all spectra with highlighted bins
plt.figure(figsize=(10, 6))

# Number of integrations to plot (can be reduced to avoid overcrowding)
plot_every = max(1, len(calibrated_optspec) // 20)  # Plot every Nth integration
cmap_integrations = plt.get_cmap('viridis_r', len(calibrated_optspec) // plot_every)

# Plot a subset of the integrations for better visibility
for idx, integration in enumerate(range(0, len(calibrated_optspec), plot_every)):
    plt.plot(
        eureka_wave_1d,
        calibrated_optspec[integration],
        color=cmap_integrations(idx / (len(calibrated_optspec) // plot_every)),
        linewidth=1,
        alpha=0.5
    )

# Add shaded regions for each bin with the same colors
for i in range(num_bins):
    bin_start = bin_edges[i]
    bin_end = bin_edges[i + 1]
    plt.axvspan(bin_start, bin_end, alpha=0.2, color=colors_bins[i], 
                label=f'Bin {i + 1}: {bin_centers[i]:.2f} μm')

plt.xlabel("Wavelength (microns)")
plt.ylabel("Flux")
plt.title(f"All Integrations with Highlighted Wavelength Bins")
plt.grid(alpha=0.3)
plt.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()